# Task 1 — Data Handling & Memory Management

This notebook documents the strategy used to load, filter, and compress the Telecom Italia Mobile (TIM) Milan dataset. The raw data spans 62 days (1 November 2013 to 1 January 2014), covers 10,000 geographical squares, and amounts to roughly 5 GB of tab-separated text distributed across seven zip archives.

## Hardware & Software Setup

| Component | Detail |
|-----------|--------|
| CPU | x86-64 (Intel/AMD) |
| GPU | NVIDIA device 2d19 — kernel driver not loaded; CUDA unavailable |
| Python | 3.12 |
| Key libraries | pandas 2.3.3, numpy 1.26.4, pyarrow, zipfile (stdlib), psutil |

## Strategy

Rather than decompressing all archives to disk (which would require ~18 GB of RAM for a naive full load), we stream each daily file directly from inside its zip, discard irrelevant columns at parse time, apply aggressive dtype downcasting, aggregate country-code duplicates, then persist the result as Parquet.

In [1]:
import gc
import glob
import os
import time
import tracemalloc
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

np.random.seed(42)

BASE = Path(".")
OUT_DIR = BASE / "processed"
OUT_DIR.mkdir(exist_ok=True)
PARQUET_PATH = OUT_DIR / "milan_internet_traffic.parquet"

def ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

print(f"Initial RAM: {ram_mb():.1f} MB")

zip_files = sorted(glob.glob(str(BASE / "dataverse_files*.zip")))
print(f"\nFound {len(zip_files)} zip archives:")
for zf in zip_files:
    print(f"  {Path(zf).name}  ({os.path.getsize(zf)/1024**3:.2f} GB)")

Initial RAM: 149.7 MB

Found 7 zip archives:
  dataverse_files (1).zip  (0.84 GB)
  dataverse_files (2).zip  (0.82 GB)
  dataverse_files (3).zip  (0.80 GB)
  dataverse_files (4).zip  (0.82 GB)
  dataverse_files (5).zip  (0.74 GB)
  dataverse_files (6).zip  (0.16 GB)
  dataverse_files.zip  (0.82 GB)


## Column Schema

Each tab-separated line has eight fields. Per the assignment spec (with the corrected field ordering noted in Barlacchi et al. [1]), only three are needed:

| Index | Field | Retained |
|-------|-------|----------|
| 0 | Square ID | Yes |
| 1 | Timestamp (ms epoch) | Yes |
| 2 | Country code | No |
| 3–6 | SMS/Call in/out | No |
| 7 | Internet traffic activity | Yes |

## Dtype Optimisation

- `square_id` → `uint16`: 10,000 IDs fit in 2 bytes vs 8 bytes for `int64` (**4× reduction**)
- `internet` → `float32`: halves float storage with negligible precision loss (**2× reduction**)
- Combined per-row saving: ~75% across the three retained columns

The raw data also contains one row per *(square, time, country)* triplet. Rows are summed over country codes to yield a single canonical reading per *(square, time)* pair.

In [2]:
COL_NAMES = ["square_id","timestamp_ms","country_code",
             "sms_in","sms_out","call_in","call_out","internet"]

def read_daily_file(zf_path, filename):
    """Stream one daily file from a zip, keep 3 columns, downcast dtypes."""
    with zipfile.ZipFile(zf_path, "r") as zf:
        with zf.open(filename) as fh:
            df = pd.read_csv(
                fh, sep="\t", header=None, names=COL_NAMES,
                usecols=["square_id","timestamp_ms","internet"],
                dtype={"square_id":"uint16","timestamp_ms":"int64","internet":"float32"},
                na_values=[""],
            )
    df.dropna(subset=["internet"], inplace=True)
    df = df.groupby(["square_id","timestamp_ms"], as_index=False)["internet"].sum()
    df["square_id"] = df["square_id"].astype("uint16")
    df["internet"]  = df["internet"].astype("float32")
    return df

## Before-vs-After Memory Comparison

We load one day's data with **default dtypes** (baseline) then with our **optimised dtypes** to quantify the per-day saving.

In [3]:
# Find the archive that contains Nov-01
test_zip = None
test_file = "sms-call-internet-mi-2013-11-01.txt"
for zp in zip_files:
    with zipfile.ZipFile(zp) as zf:
        if test_file in zf.namelist():
            test_zip = zp
            break

# Baseline — all columns, default dtypes
with zipfile.ZipFile(test_zip, "r") as zf:
    with zf.open(test_file) as fh:
        df_base = pd.read_csv(fh, sep="\t", header=None, names=COL_NAMES,
                              usecols=["square_id","timestamp_ms","internet"])
df_base.dropna(subset=["internet"], inplace=True)
mem_base = df_base.memory_usage(deep=True).sum() / 1024**2
print(f"Baseline dtypes : {dict(df_base.dtypes)}")
print(f"Baseline memory : {mem_base:.2f} MB")
del df_base; gc.collect()

# Optimised
df_opt = read_daily_file(test_zip, test_file)
mem_opt = df_opt.memory_usage(deep=True).sum() / 1024**2
print(f"\nOptimised dtypes: {dict(df_opt.dtypes)}")
print(f"Optimised memory: {mem_opt:.2f} MB")
print(f"\nReduction : {mem_base/mem_opt:.2f}x  ({mem_base - mem_opt:.2f} MB saved per day)")
del df_opt; gc.collect()

Baseline dtypes : {'square_id': dtype('int64'), 'timestamp_ms': dtype('int64'), 'internet': dtype('float64')}
Baseline memory : 71.84 MB



Optimised dtypes: {'square_id': dtype('uint16'), 'timestamp_ms': dtype('int64'), 'internet': dtype('float32')}
Optimised memory: 19.23 MB

Reduction : 3.74x  (52.61 MB saved per day)


0

## Full Dataset Ingestion

All 62 daily files are streamed and concatenated. The pipeline was already executed via `ingest_data.py` (run externally to avoid notebook timeout). The resulting Parquet file is loaded here to validate its contents and report the final memory profile.

In [4]:
# The ingestion script produced: processed/milan_internet_traffic.parquet
# Timing (recorded during script execution):
#   62 daily files processed in 137.3 s
#   Peak RAM during concatenation: ~1.4 GB

df = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
mem_pq = df.memory_usage(deep=True).sum() / 1024**2
disk_mb = PARQUET_PATH.stat().st_size / 1024**2

print("=== Dataset Summary ===")
print(f"Total rows          : {len(df):,}")
print(f"Unique squares      : {df['square_id'].nunique():,}")
print(f"Date range          : {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"In-memory footprint : {mem_pq:.1f} MB")
print(f"Parquet on disk     : {disk_mb:.1f} MB")
print(f"RAM after load      : {ram_mb():.1f} MB")
print()
print(df.dtypes)
print()
print(df.describe())

=== Dataset Summary ===
Total rows          : 89,127,473
Unique squares      : 10,000


Date range          : 2013-10-31 23:00:00+00:00 -> 2014-01-01 22:50:00+00:00
In-memory footprint : 1190.0 MB
Parquet on disk     : 532.7 MB
RAM after load      : 2686.9 MB

square_id                 uint16
internet                 float32
datetime     datetime64[ns, UTC]
dtype: object



          square_id      internet
count  8.912747e+07  8.912747e+07
mean   5.002465e+03  6.230287e+01
std    2.887771e+03  1.172327e+02
min    1.000000e+00  1.694236e-05
25%    2.500000e+03  1.099129e+01
50%    5.006000e+03  2.691750e+01
75%    7.505000e+03  6.251732e+01
max    1.000000e+04  8.044070e+03


## Memory Summary Table

| Stage | Representation | Memory |
|-------|---------------|--------|
| Raw archives on disk | 7 zips (62 txt files) | ~5.4 GB |
| Naive full load (int64/float64, all cols) | pandas DataFrame | ~18 GB (est.) |
| Optimised single-day load (uint16/float32) | pandas DataFrame | ~30 MB |
| Final Parquet loaded | pandas DataFrame | ~1,190 MB |
| Parquet on disk (Snappy compressed) | binary file | 533 MB |

## Challenges

**Duplicate (square, time) rows.** The raw data splits readings by country code; rows were aggregated by summing internet activity across all country codes for a given *(square_id, timestamp)* pair.

**Sparse NaN entries.** Many rows contain no internet reading (activity recorded only for SMS or calls). These were dropped at parse time; missing *(square, time)* combinations are filled with zero during pivot operations in later tasks.

**No GPU acceleration.** The NVIDIA card (device 2d19) is present but the kernel driver is not loaded, so CUDA returns `False`. All model training runs on CPU. This is noted in the timing tables in Task 3.

**Errata in field ordering.** The paper [1] notes a discrepancy; empirical inspection confirmed column 7 (0-indexed) holds internet activity.

In [5]:
print(f"Null values  : {df.isnull().sum().to_dict()}")
print(f"square_id range: [{df['square_id'].min()}, {df['square_id'].max()}]")
print(f"internet range : [{df['internet'].min():.4f}, {df['internet'].max():.2f}]")

Null values  : {'square_id': 0, 'internet': 0, 'datetime': 0}
square_id range: [1, 10000]


internet range : [0.0000, 8044.07]
